# California Buritto(Burger Town) Analytics Dashboard: Data Exploration and Preparation

**Goal:** Check the raw Excel file for quality problems and prepare it for loading into MySQL.

**Dataset:** `data.xlsx` has one row per *line item*. One order (`BillNo`) can have several rows. So I must be careful about what "a record" means: 300,000 rows is not 300,000 orders.

**Plan for this notebook:**
1. Read the file and confirm its size.
2. Check for missing values, duplicates, and odd values.
3. Check that the order-level columns are consistent inside each order.
4. Save a compressed CSV, so the web app never needs to read the slow Excel file.

## 1. Read the Excel file

I read the Excel file once and convert it to CSV. Reading a 13 MB `.xlsx` takes about a minute, which is too slow for a web app. A compressed CSV loads in under a second.

**Expected shape:** 300,000 rows and 10 columns.

In [16]:
import pandas as pd
RAW = "../data/raw/data.xlsx"
OUT = "../data/orders.csv.gz"
df = pd.read_excel(RAW, engine="openpyxl")
df.shape

(300000, 10)

## 2. Data quality checks

I checked the points that could make my dashboard numbers wrong:

- **Missing values and duplicates:** none found.
- **Orders vs rows:** there are 110,478 unique orders but 300,000 rows. The dashboard must show both, and must count orders with `COUNT(DISTINCT BillNo)`, not `COUNT(*)`.
- **Date range:** 17 Jun 2025 to 16 Jun 2026. The first and last months are partial, so a monthly trend will show a false drop at both ends. I will label this on the chart.
- **Revenue:** price x quantity, summed to ₹69,480,952. I assume ₹, because the file has no currency column. I will use this total to test my dashboard.
- **Free items:** a few rows have a price of 0. They are the free dips (`Dip - BBQ`, `Dip - Mayo`). I decided to **keep them**. They are real items sold, so they count as line items and sold quantity, and they add ₹0 revenue.

In [17]:
print("missing values:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())
print("orders:", df.BillNo.nunique())
print("dates:", df.Order_Datetime.min(), "to", df.Order_Datetime.max())
print("revenue:", (df.Price * df.Quantity).sum())
print("free items:", df[df.Price == 0].Item.unique())

missing values: 0
duplicate rows: 0
orders: 110478
dates: 2025-06-17 11:02:40 to 2026-06-16 23:48:34
revenue: 69480952
free items: <StringArray>
['Dip - Mayo', 'Dip - BBQ']
Length: 2, dtype: str


## 3. Are order-level columns consistent?

Outlet, time, order type and payment method should be the same on every row of one order. If they are not, grouping by order would give wrong answers.

**Result:** every order has exactly one value for each of these columns. So it is safe to count orders by `BillNo`, and to filter by outlet, order type or payment method at the row level without splitting an order.

In [18]:
g = df.groupby("BillNo")
for c in ["Outlet_Name", "Order_Datetime", "Order_Type", "Settlement"]:
    print(c, (g[c].nunique() > 1).sum())

Outlet_Name 0
Order_Datetime 0
Order_Type 0
Settlement 0


## 4. Save the cleaned data

I save the data as a gzip-compressed CSV (about 4 MB). I do not commit the original Excel file. It is large, and the CSV is the faster, smaller input for the loader.

I reload the saved file and check that the shape and total revenue are the same as before. This proves the save did not change any data.

In [19]:
df.to_csv(OUT, index=False)
chk = pd.read_csv(OUT, parse_dates=["Order_Datetime"])
print(chk.shape, (chk.Price * chk.Quantity).sum())

(300000, 10) 69480952


## 5. Other observations that affect the dashboard design

- **`Brand`** has only one value, so a brand filter would be useless. I use outlet as the region filter.
- **Delivery with "Cash/Card/Coupon":** some delivery rows are paid this way, even though most delivery uses Swiggy or Zomato. I kept the data as given and noted it as an assumption in the README.
- **Short February** has fewer days, which explains its lower total. It is not a business problem.

In [20]:
print("brands:", df.Brand.nunique())
print(pd.crosstab(df.Settlement, df.Order_Type))
monthly = df.assign(month=df.Order_Datetime.dt.to_period("M")).groupby("month").agg(
    rows=("BillNo", "size"), orders=("BillNo", "nunique"))
monthly

brands: 1
Order_Type        Delivery  Dine-In  Takeaway
Settlement                                   
Cash/Card/Coupon     10184   124328     55447
Dineout                  0    11044      4783
SwiggyPay            47022        0         0
ZomatoPay            47192        0         0


,rows,orders
month,,
2025-06,11648,4277
2025-07,25402,9327
2025-08,25235,9280
2025-09,25245,9237
2025-10,25769,9437
2025-11,24467,9029
2025-12,25341,9357
2026-01,25366,9302
2026-02,22747,8423


## Summary

- The data is clean: no missing values or duplicates.
- Orders and line items are different counts: 110,478 orders vs 300,000 rows.
- Known quirks: partial first and last months, free dips, single brand.
- Reference totals for testing the dashboard: revenue ₹69,480,952, 434,448 items sold, average order value ₹628.91.
- **Next:** load this data into MySQL with an explicit schema and indexes, then build the API and the dashboard.